In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# ================== Paths ==================
path_few  = r"F:\Thesis\project\1-BaselineTest\Models Responses\few-shot\gpt oss 120\results_few_shot.csv"
path_zero = r"F:\Thesis\project\1-BaselineTest\Models Responses\zero-shot\gpt oss 120\results_zero_shot.csv"
path_gold = r"F:\Thesis\project\1-BaselineTest\GOLD\Gold.csv"

model_id_col  = "id"
model_ans_col = "answer"
gold_id_col   = "idx"
gold_ans_col  = "Gold"

# ================== Helpers ==================
def norm_ans(x):
    if pd.isna(x): return None
    s = str(x).strip().strip('"').strip("'")
    try:
        v = int(float(s))
        return str(v) if v in {1,2,3,4} else None
    except:
        for d in ["1","2","3","4"]:
            if s.endswith(d): return d
        return None

def parse_gold_single_or_pair(x):
    if pd.isna(x): 
        return set()
    s = str(x).strip()
    if "-" in s:
        parts = [p.strip() for p in s.split("-")]
        return {norm_ans(p) for p in parts if norm_ans(p) is not None}
    v = norm_ans(s)
    return {v} if v is not None else set()

def build_eval(model_path, gold_df, tag):
    m = pd.read_csv(model_path)
    m[model_ans_col] = m[model_ans_col].apply(norm_ans)
    has_conf = "confidence" in m.columns
    has_lat  = "latency_ms" in m.columns

    g = gold_df.copy()
    g["gold_set"] = g[gold_ans_col].apply(parse_gold_single_or_pair)
    g["is_multi"] = g["gold_set"].apply(lambda s: len(s) > 1)

    df = pd.merge(
        m, g[[gold_id_col, "gold_set", "is_multi"]],
        left_on=model_id_col, right_on=gold_id_col, how="inner"
    )
    df["correct"] = df.apply(lambda r: (r[model_ans_col] in r["gold_set"]) if r[model_ans_col] is not None else False, axis=1).astype(int)
    df["run_type"] = tag
    return df, has_conf, has_lat

# ================== Load both runs ==================
gold = pd.read_csv(path_gold)
df_zero, has_conf_zero, has_lat_zero = build_eval(path_zero, gold, "zero_shot")
df_few,  has_conf_few,  has_lat_few  = build_eval(path_few,  gold, "few_shot")

# ================== Save per-run reports ==================
def per_run_reports(df, tag):
    root = Path(f"eval_report_{tag}")
    root.mkdir(parents=True, exist_ok=True)
    sns.set_style("whitegrid")
    plt.rcParams['font.family'] = 'Arial'

    # 1) Accuracy
    acc_dir = root / "eval_accuracy"; acc_dir.mkdir(exist_ok=True)
    acc = df["correct"].mean()*100 if len(df) else 0.0
    (acc_dir / "accuracy_overall.txt").write_text(f"Accuracy: {acc:.2f}%\n", encoding="utf-8")

    by_pred = df.groupby(model_ans_col)["correct"].agg(["mean","count"]).reset_index()
    by_pred["accuracy_%"] = by_pred["mean"]*100
    by_pred.drop(columns=["mean"]).to_csv(acc_dir / "accuracy_by_pred.csv", index=False, encoding="utf-8-sig")

    conf = pd.crosstab(
        df[model_ans_col],
        df["gold_set"].apply(lambda s: sorted(list(s))[0] if len(s)>0 else None),
        rownames=["pred"], colnames=["gold_primary"], dropna=False
    ).fillna(0).astype(int)
    conf.to_csv(acc_dir / "confusion.csv", encoding="utf-8-sig")
    df.to_csv(acc_dir / "eval_detailed.csv", index=False, encoding="utf-8-sig")

    # Plots
    plt.figure(figsize=(4,4))
    plt.bar(["Accuracy"], [acc], color="#4E79A7"); plt.ylim(0,100)
    plt.title(f"Accuracy ({tag})"); plt.ylabel("Accuracy (%)")
    plt.text(0, acc+1, f"{acc:.1f}%", ha="center", va="bottom", fontweight="bold")
    plt.tight_layout(); plt.savefig(acc_dir / "plot_accuracy_overall.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

    tmp = by_pred.copy()
    tmp[model_ans_col] = tmp[model_ans_col].astype(str)
    tmp = tmp.sort_values(model_ans_col)
    plt.figure(figsize=(6,4))
    plt.bar(tmp[model_ans_col], tmp["accuracy_%"], color="#F28E2B"); plt.ylim(0,100)
    plt.xlabel("Predicted option"); plt.ylabel("Accuracy (%)"); plt.title(f"Accuracy by Predicted Option ({tag})")
    for x,y in zip(tmp[model_ans_col], tmp["accuracy_%"]): plt.text(x, y+1, f"{y:.1f}%", ha="center", va="bottom", fontsize=9)
    plt.tight_layout(); plt.savefig(acc_dir / "plot_accuracy_by_pred.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

    plt.figure(figsize=(5,4))
    sns.heatmap(conf, annot=True, fmt="d", cmap="Blues", cbar=False, linewidths=1, linecolor="white")
    plt.title(f"Confusion Matrix ({tag})")
    plt.tight_layout(); plt.savefig(acc_dir / "plot_confusion_heatmap.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

    # 2) Calibration (if confidence exists)
    cal_dir = root / "calibration_confidence"; cal_dir.mkdir(exist_ok=True)
    if "confidence" in df.columns and df["confidence"].notna().any():
        calib = df.groupby("confidence")["correct"].agg(["mean","count"]).reset_index()
        calib.rename(columns={"mean":"accuracy"}, inplace=True); calib["accuracy_%"] = calib["accuracy"]*100
        calib.to_csv(cal_dir / "calibration_by_conf.csv", index=False, encoding="utf-8-sig")

        plt.figure(figsize=(5,4))
        plt.plot(calib["confidence"], calib["accuracy_%"], marker="o", color="#4E79A7", label="Observed")
        plt.plot([1,5],[20,100], linestyle="--", color="gray", label="Ideal (scaled)")
        plt.xticks([1,2,3,4,5]); plt.ylim(0,100)
        plt.xlabel("Confidence"); plt.ylabel("Accuracy (%)"); plt.title(f"Reliability Curve ({tag})"); plt.legend()
        plt.tight_layout(); plt.savefig(cal_dir / "plot_reliability_curve.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

        rows=[]
        for t in [1,2,3,4,5]:
            sub = df[df["confidence"]>=t]
            coverage = len(sub)/len(df)*100 if len(df) else 0.0
            acc_t = sub["correct"].mean()*100 if len(sub) else np.nan
            rows.append({"threshold":t,"coverage_%":coverage,"accuracy_%":acc_t,"n":len(sub)})
        pd.DataFrame(rows).to_csv(cal_dir / "threshold_tradeoff.csv", index=False, encoding="utf-8-sig")

    # 3) Latency (if available)
    lat_dir = root / "latency_profile"; lat_dir.mkdir(exist_ok=True)
    if "latency_ms" in df.columns and df["latency_ms"].notna().any():
        lat_summary = df["latency_ms"].agg(["mean","median","min","max",
                                            lambda s: np.percentile(s.dropna(),90),
                                            lambda s: np.percentile(s.dropna(),95)]).round(1)
        lat_summary.index = ["mean","median","min","max","p90","p95"]
        lat_summary.to_csv(lat_dir / "latency_summary.csv", header=False, encoding="utf-8-sig")

        plt.figure(figsize=(6,4))
        plt.hist(df["latency_ms"].dropna(), bins=20, color="#4E79A7", edgecolor="black")
        plt.xlabel("Latency (ms)"); plt.ylabel("Frequency"); plt.title(f"Latency Distribution ({tag})")
        plt.tight_layout(); plt.savefig(lat_dir / "plot_latency_dist.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

        # accuracy vs latency bucket
        bins = [0, 1000, 2000, 3000, 5000, 10000, np.inf]
        labels = ["<=1s","1-2s","2-3s","3-5s","5-10s",">10s"]
        df["lat_bucket"] = pd.cut(df["latency_ms"], bins=bins, labels=labels, right=True)
        acc_by_bucket = df.groupby("lat_bucket")["correct"].mean().mul(100).reset_index()
        acc_by_bucket.to_csv(lat_dir / "accuracy_by_latency_bucket.csv", index=False, encoding="utf-8-sig")

        plt.figure(figsize=(7,4))
        plt.bar(acc_by_bucket["lat_bucket"].astype(str), acc_by_bucket["correct"], color="#76B7B2")
        plt.ylim(0,100); plt.xlabel("Latency bucket"); plt.ylabel("Accuracy (%)"); plt.title(f"Accuracy by Latency Bucket ({tag})")
        for x,y in zip(acc_by_bucket["lat_bucket"].astype(str), acc_by_bucket["correct"]):
            plt.text(x, y+1, f"{y:.1f}%", ha="center", va="bottom", fontsize=9)
        plt.tight_layout(); plt.savefig(lat_dir / "plot_accuracy_by_latency_bucket.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

    # 4) Errors
    err_dir = root / "error_inspection"; err_dir.mkdir(exist_ok=True)
    err_pairs = conf.stack().reset_index(name="count").rename(columns={"pred":model_ans_col, "gold_primary":"gold"})
    err_pairs = err_pairs[err_pairs[model_ans_col] != err_pairs["gold"]].sort_values("count", ascending=False)
    err_pairs.head(20).to_csv(err_dir / "top_mistakes.csv", index=False, encoding="utf-8-sig")

    if "confidence" in df.columns and df["confidence"].notna().any():
        hard_false = df[(df["correct"]==0) & (df["confidence"]>=4)]
        lucky_true = df[(df["correct"]==1) & (df["confidence"]<=2)]
        hard_false.to_csv(err_dir / "review_hard_false_high_conf.csv", index=False, encoding="utf-8-sig")
        lucky_true.to_csv(err_dir / "review_lucky_true_low_conf.csv", index=False, encoding="utf-8-sig")

    return root

root_zero = per_run_reports(df_zero, "zero_shot")
root_few  = per_run_reports(df_few,  "few_shot")

# ================== Compare two runs (direct head-to-head) ==================
cmp_root = Path("eval_compare_zero_vs_few")
cmp_root.mkdir(exist_ok=True)

# انتخاب ستون‌های کلیدی موجود
keep_cols = [c for c in ["id","answer","confidence","latency_ms","correct"] if c in df_zero.columns]
zero_cmp = df_zero[keep_cols].copy().rename(columns={
    "answer":"answer_zero","confidence":"conf_zero","latency_ms":"lat_zero","correct":"correct_zero"
})

keep_cols = [c for c in ["id","answer","confidence","latency_ms","correct"] if c in df_few.columns]
few_cmp  = df_few[keep_cols].copy().rename(columns={
    "answer":"answer_few","confidence":"conf_few","latency_ms":"lat_few","correct":"correct_few"
})

cmp = pd.merge(zero_cmp, few_cmp, on="id", how="inner")

# اختلاف‌ها
cmp["same_answer"]   = (cmp["answer_zero"] == cmp["answer_few"]).astype(int)
cmp["gain_correct"]  = ((cmp["correct_zero"]==0) & (cmp["correct_few"]==1)).astype(int)
cmp["loss_correct"]  = ((cmp["correct_zero"]==1) & (cmp["correct_few"]==0)).astype(int)
if "lat_zero" in cmp.columns and "lat_few" in cmp.columns:
    cmp["lat_delta_ms"] = cmp["lat_few"] - cmp["lat_zero"]

cmp.to_csv(cmp_root / "compare_zero_vs_few.csv", index=False, encoding="utf-8-sig")

# خلاصه‌ مقایسه
summary = {
    "n_common": len(cmp),
    "acc_zero_%": cmp["correct_zero"].mean()*100 if len(cmp) else np.nan,
    "acc_few_%":  cmp["correct_few"].mean()*100 if len(cmp) else np.nan,
    "same_answer_%": cmp["same_answer"].mean()*100 if len(cmp) else np.nan,
    "gain_%": cmp["gain_correct"].mean()*100 if len(cmp) else np.nan,
    "loss_%": cmp["loss_correct"].mean()*100 if len(cmp) else np.nan,
}
pd.Series(summary).to_csv(cmp_root / "summary.csv", header=False, encoding="utf-8-sig")

# نمودار اختلاف دقت
plt.figure(figsize=(5,4))
vals = [summary["acc_zero_%"], summary["acc_few_%"]]
plt.bar(["zero_shot","few_shot"], vals, color=["#4E79A7","#F28E2B"])
for i,v in enumerate(vals): plt.text(i, v+1, f"{v:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")
plt.ylim(0,100); plt.ylabel("Accuracy (%)"); plt.title("Accuracy: Zero vs Few")
plt.tight_layout(); plt.savefig(cmp_root / "plot_accuracy_zero_vs_few.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

# نمودار تغییر تاخیر (در صورت وجود هر دو)
if "lat_delta_ms" in cmp.columns and cmp["lat_delta_ms"].notna().any():
    plt.figure(figsize=(6,4))
    plt.hist(cmp["lat_delta_ms"].dropna(), bins=30, color="#76B7B2", edgecolor="black")
    plt.axvline(0, color="red", linestyle="--")
    plt.xlabel("Latency delta (few - zero) ms"); plt.title("Latency change (few minus zero)")
    plt.tight_layout(); plt.savefig(cmp_root / "plot_latency_delta_hist.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

print("✅ Reports saved:",
      f"\n - {root_zero}",
      f"\n - {root_few}",
      f"\n - {cmp_root}")


C:\Users\sazgar\AppData\Local\Temp\ipykernel_1724\686067361.py:147: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  acc_by_bucket = df.groupby("lat_bucket")["correct"].mean().mul(100).reset_index()
C:\Users\sazgar\AppData\Local\Temp\ipykernel_1724\686067361.py:147: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  acc_by_bucket = df.groupby("lat_bucket")["correct"].mean().mul(100).reset_index()


✅ Reports saved: 
 - eval_report_zero_shot 
 - eval_report_few_shot 
 - eval_compare_zero_vs_few
